# Particle (DEM) simulation of a wheel climbing sand slopes, for a free Kaggle or Colab GPU

**Status: prepared 2026-10-21, NOT RUN in this repository** (the cloud machine of the session had no GPU). Huda runs it herself on a free GPU (Kaggle: Settings, Accelerator, GPU T4; Colab: Runtime, Change runtime type, T4 GPU). Everything it produces is **SIMULATED BY ME** (simulation only, model assumption dependent).

What it does: a VIPER type rigid wheel (mesh shipped with DEM-Engine), driven at a fixed angular speed with a fixed load, on a bed of polydisperse spheres, at slopes 0 to 25 degrees by tilting gravity, the method of the DEM-Engine demo `DEMdemo_WheelSlopeSlip.cpp` (projectchrono/DEM-Engine, BSD 3 licence) with a smaller sphere bed so it fits a free GPU. Slip is 1 minus the mean forward speed over the wheel's rim speed, averaged after the wheel has settled.

Free software only: the `deme` package from PyPI (DEM-Engine Python bindings, version 3.0.19 at the time of writing). No account other than the free Kaggle or Colab account.

Afterwards: download `chrono_dem_results.csv`, put it in the repository as `data/raw/chrono_dem/chrono_dem_results.csv`, and run `python3 scripts/ingest_chrono_dem.py`.

In [ ]:
# 1. Check the GPU and install DEM-Engine (about 1 to 3 minutes; the first kernel compile later can take much longer)
!nvidia-smi
!pip install -q deme==3.0.19
import deme, math, time, csv, platform
print("deme imported; python", platform.python_version())

In [ ]:
# 2. Parameters (change only here). Values are ASSUMPTIONS unless marked.
SLOPES_DEG = [0.0, 5.0, 10.0, 15.0, 20.0, 25.0]
GRAIN_R = 0.006            # mean sphere radius, m (coarse for a free GPU; real sand is about 0.0002 m)
GRAIN_DENSITY = 2600.0     # kg/m3, quartz
BED = (1.6, 0.40, 0.16)    # bed length, width, depth, m
WHEEL_MASS = 5.0           # kg (as in DEMdemo_WheelSlopeSlip)
TOTAL_LOAD_KG = 50.0       # total load on the wheel in kg (the demo uses 110; 50 keeps a light rover in mind)
W_R = 0.8                  # wheel angular speed, rad/s (as in the demo)
STEP = 2e-6                # time step, s (as in the demo)
SETTLE_T, SINK_T, RUN_T, AVG_FROM = 0.5, 0.2, 4.0, 2.0   # s
MU_GRAIN, MU_WHEEL = 0.5, 0.8                              # friction (as in the demo)
TIME_BUDGET_H = 9.0        # stop before the free session ends (Kaggle sessions last up to 12 h)
print("parameters set")

In [ ]:
# 3. One run at one slope. Returns the slip and timings.
def run_slope(slope_deg, seed=20261021):
    import random
    G = 9.81; th = math.radians(slope_deg)
    s = deme.DEMSolver()
    s.SetVerbosity("ERROR")
    m_wheel = s.LoadMaterial({"E": 1e9, "nu": 0.3, "CoR": 0.5, "mu": 0.5, "Crr": 0.0})
    m_sand = s.LoadMaterial({"E": 1e9, "nu": 0.3, "CoR": 0.5, "mu": MU_GRAIN, "Crr": 0.0})
    s.SetMaterialPropertyPair("mu", m_wheel, m_sand, MU_WHEEL)
    L, Wd, D = BED
    s.InstructBoxDomainDimension([-L / 2, L / 2], [-Wd / 2, Wd / 2], [-D, 1.0])
    s.InstructBoxDomainBoundingBC("top_open", m_sand)
    radii = [GRAIN_R * (0.85 + 0.06 * i) for i in range(6)]
    templates = [s.LoadSphereType(GRAIN_DENSITY * 4 / 3 * math.pi * r ** 3, r, m_sand) for r in radii]
    rmax = max(radii)
    pts = deme.DEMBoxHCPSampler([0, 0, -D / 2], [L / 2 - rmax * 1.1, Wd / 2 - rmax * 1.1, D / 2 - rmax * 1.1], 2.01 * rmax)
    rng = random.Random(seed)
    bed = s.AddClumps([rng.choice(templates) for _ in pts], pts); bed.SetFamily(0)
    wheel = s.AddWavefrontMeshObject(deme.GetDEMEDataFile("mesh/rover_wheels/viper_wheel_right.obj"), m_wheel)
    wheel_rad, wheel_w = 0.25, 0.2   # VIPER type wheel mesh as used by the DEM-Engine demos (radius 0.25 m)
    wheel.SetMass(WHEEL_MASS)
    wheel.SetMOI([WHEEL_MASS / 12 * (3 * wheel_rad ** 2 + wheel_w ** 2), WHEEL_MASS * wheel_rad ** 2 / 2, WHEEL_MASS / 12 * (3 * wheel_rad ** 2 + wheel_w ** 2)])
    wheel.SetFamily(10)
    tr = s.Track(wheel)
    added = (TOTAL_LOAD_KG - WHEEL_MASS) * G
    s.SetFamilyPrescribedAngVel(1, "0", str(W_R), "0", False)
    s.AddFamilyPrescribedAcc(1, str(-added * math.sin(th) / WHEEL_MASS), "none", str(-added * math.cos(th) / WHEEL_MASS))
    s.SetFamilyFixed(10)
    top = s.CreateInspector("clump_max_z")
    s.SetGravitationalAcceleration([-G * math.sin(th), 0, -G * math.cos(th)])   # uphill is +x
    s.SetTimeStepSize(STEP); s.SetCDUpdateFreq(15); s.SetMaxVelocity(30.0); s.SetErrorOutVelocity(60.0)
    t0 = time.time(); s.Initialize(); t_init = time.time() - t0
    s.DoDynamicsThenSync(SETTLE_T)                       # settle the bed under the tilted gravity
    tr.SetPos([-L / 2 + 0.35, 0, top.GetValue() + wheel_rad + 0.002])
    s.DoDynamicsThenSync(SINK_T)                         # let the fixed wheel family rest
    s.ChangeFamily(10, 1)                                # release: driven and loaded
    xs, t = [], 0.0
    while t < RUN_T:
        s.DoDynamicsThenSync(0.05); t += 0.05
        if t >= AVG_FROM:
            xs.append((t, tr.Pos()[0]))
    v = (xs[-1][1] - xs[0][1]) / (xs[-1][0] - xs[0][0])
    return {"slope_deg": slope_deg, "slip": round(1 - v / (W_R * wheel_rad), 4), "mean_vx_m_s": round(v, 5), "wheel_radius_m": wheel_rad,
            "w_r_rad_s": W_R, "total_load_kg": TOTAL_LOAD_KG, "n_spheres": len(pts), "grain_r_m": GRAIN_R, "run_t_s": RUN_T,
            "init_wall_s": round(t_init, 1), "wall_s": round(time.time() - t0, 1), "deme_version": "3.0.19"}
print("function defined")

In [ ]:
# 4. Time one short run first (smoke test), then decide whether the full sweep fits in the time budget.
RUN_T_SAVE = RUN_T; RUN_T = AVG_FROM + 0.3
t0 = time.time(); r = run_slope(0.0); RUN_T = RUN_T_SAVE
print(r); per_run_h = (time.time() - t0) * (RUN_T / (AVG_FROM + 0.3)) / 3600
print(f"estimated hours per full slope run: {per_run_h:.2f}; for {len(SLOPES_DEG)} slopes: {per_run_h * len(SLOPES_DEG):.1f} h")

In [ ]:
# 5. Full sweep, written after every slope so nothing is lost if the session ends.
rows, start = [], time.time()
for sl in SLOPES_DEG:
    if (time.time() - start) / 3600 > TIME_BUDGET_H:
        print("time budget reached; stopping before", sl); break
    r = run_slope(sl); rows.append(r); print(r)
    with open("chrono_dem_results.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("done; download chrono_dem_results.csv from the file panel")